# Boston 住宅価格データセットの探索的データ分析（EDA）

このノートブックでは、Boston 住宅価格データセットを詳しく調べます。

## 1. 日本語フォントの自動検出

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

def setup_japanese_font():
    """日本語フォントを自動検出して設定"""
    system = platform.system()
    
    # 候補フォントリスト（優先順）
    font_candidates = [
        'MS Gothic',
        'Yu Gothic',
        'Meiryo',
        'Hiragino Sans',
        'Hiragino Kaku Gothic Pro',
        'Noto Sans CJK JP',
        'IPAexGothic',
        'IPAPGothic'
    ]
    
    # 利用可能なフォントを取得
    available_fonts = [f.name for f in fm.fontManager.ttflist]
    
    # 候補から最初に見つかったフォントを使用
    for font in font_candidates:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            print(f'日本語フォント設定: {font}')
            return font
    
    print('警告: 日本語フォントが見つかりませんでした')
    return None

# フォント設定
setup_japanese_font()
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の文字化け防止

## 2. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Seaborn のスタイル設定
sns.set_style('whitegrid')
sns.set_palette('husl')

print('ライブラリのインポート完了')

## 3. データの読み込み

In [ ]:
# データを読み込む
df = pd.read_csv('../data/Boston.csv')

# 使用する列のみを抽出
df_model = df[['CRIME', 'RM', 'LSTAT', 'PTRATIO', 'PRICE']].copy()

print(f'データ件数: {len(df_model)} 件')
print(f'特徴量数: {len(df_model.columns) - 1} 個')
print(f'\nカラム:\n{df_model.columns.tolist()}')

df_model.head(10)

## 4. 基本統計量

In [ ]:
print('=== 数値特徴量の基本統計量 ===')
df_model[['RM', 'LSTAT', 'PTRATIO', 'PRICE']].describe()

In [ ]:
print('=== CRIME カテゴリの分布 ===')
print(df_model['CRIME'].value_counts())
print(f'\n割合:')
print(df_model['CRIME'].value_counts(normalize=True) * 100)

## 5. 欠損値の確認

In [ ]:
print('=== 欠損値の確認 ===')
missing = df_model.isnull().sum()
missing_pct = (missing / len(df_model)) * 100

missing_df = pd.DataFrame({
    '欠損値数': missing,
    '欠損率(%)': missing_pct
})

print(missing_df[missing_df['欠損値数'] > 0])

if missing.sum() == 0:
    print('\n✓ 欠損値はありません')
else:
    print(f'\n⚠ 欠損値が {missing.sum()} 件あります')

## 6. 価格（PRICE）の分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ヒストグラム
axes[0].hist(df_model['PRICE'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('価格（千ドル）')
axes[0].set_ylabel('件数')
axes[0].set_title('住宅価格の分布')
axes[0].axvline(df_model['PRICE'].mean(), color='red', linestyle='--', label=f'平均: {df_model["PRICE"].mean():.1f}')
axes[0].legend()

# 箱ひげ図
axes[1].boxplot(df_model['PRICE'], vert=True)
axes[1].set_ylabel('価格（千ドル）')
axes[1].set_title('住宅価格の箱ひげ図')

plt.tight_layout()
plt.show()

print(f'平均価格: {df_model["PRICE"].mean():.2f} 千ドル')
print(f'中央値: {df_model["PRICE"].median():.2f} 千ドル')
print(f'最小値: {df_model["PRICE"].min():.2f} 千ドル')
print(f'最大値: {df_model["PRICE"].max():.2f} 千ドル')

## 7. CRIME カテゴリごとの価格比較

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 平均価格の棒グラフ
crime_price = df_model.groupby('CRIME')['PRICE'].mean().sort_values(ascending=False)
axes[0].bar(crime_price.index, crime_price.values, alpha=0.7)
axes[0].set_xlabel('犯罪率カテゴリ')
axes[0].set_ylabel('平均価格（千ドル）')
axes[0].set_title('犯罪率カテゴリごとの平均住宅価格')

# 箱ひげ図
crime_order = ['very_low', 'low', 'high']
axes[1].boxplot([df_model[df_model['CRIME'] == c]['PRICE'].values for c in crime_order if c in df_model['CRIME'].unique()],
                labels=[c for c in crime_order if c in df_model['CRIME'].unique()])
axes[1].set_xlabel('犯罪率カテゴリ')
axes[1].set_ylabel('価格（千ドル）')
axes[1].set_title('犯罪率カテゴリごとの価格分布')

plt.tight_layout()
plt.show()

print('=== 犯罪率カテゴリごとの統計 ===')
print(df_model.groupby('CRIME')['PRICE'].describe())

## 8. 各特徴量の分布

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('各特徴量の分布', fontsize=16)

features = ['RM', 'LSTAT', 'PTRATIO', 'PRICE']
labels = ['部屋数（RM）', '低所得者割合（LSTAT %）', '生徒-教師比率（PTRATIO）', '価格（千ドル）']

for idx, (feature, label) in enumerate(zip(features, labels)):
    row = idx // 2
    col = idx % 2
    
    axes[row, col].hist(df_model[feature], bins=20, edgecolor='black', alpha=0.7)
    axes[row, col].set_xlabel(label)
    axes[row, col].set_ylabel('件数')
    axes[row, col].axvline(df_model[feature].mean(), color='red', linestyle='--', 
                           label=f'平均: {df_model[feature].mean():.2f}')
    axes[row, col].legend()

plt.tight_layout()
plt.show()

## 9. 相関行列

In [ ]:
# CRIME をダミー変数化
df_corr = df_model.copy()
df_corr = pd.get_dummies(df_corr, columns=['CRIME'], prefix='CRIME')

# 相関行列
corr = df_corr.corr()

# ヒートマップ
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('特徴量の相関行列')
plt.tight_layout()
plt.show()

print('=== PRICE との相関（絶対値の降順） ===')
price_corr = corr['PRICE'].drop('PRICE').abs().sort_values(ascending=False)
print(price_corr)

## 10. 散布図（PRICE vs 各特徴量）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('住宅価格と各特徴量の関係', fontsize=16)

features = ['RM', 'LSTAT', 'PTRATIO']
labels = ['部屋数（RM）', '低所得者割合（LSTAT %）', '生徒-教師比率（PTRATIO）']

for idx, (feature, label) in enumerate(zip(features, labels)):
    axes[idx].scatter(df_model[feature], df_model['PRICE'], alpha=0.6)
    axes[idx].set_xlabel(label)
    axes[idx].set_ylabel('価格（千ドル）')
    
    # 近似直線を追加
    z = np.polyfit(df_model[feature], df_model['PRICE'], 1)
    p = np.poly1d(z)
    axes[idx].plot(df_model[feature], p(df_model[feature]), "r--", alpha=0.8, linewidth=2)
    
    # 相関係数を表示
    corr_val = df_model[feature].corr(df_model['PRICE'])
    axes[idx].text(0.05, 0.95, f'相関係数: {corr_val:.3f}',
                  transform=axes[idx].transAxes, fontsize=10,
                  verticalalignment='top',
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 11. 線形回帰モデルの予測性能

In [ ]:
# CRIME をダミー変数化
df_ml = df_model.copy()
df_ml = pd.get_dummies(df_ml, columns=['CRIME'], prefix='CRIME', drop_first=True)

# 特徴量と目的変数を分離
X = df_ml.drop('PRICE', axis=1)
y = df_ml['PRICE']

# 訓練データとテストデータに分割（80:20）
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 線形回帰モデルの訓練
model = LinearRegression()
model.fit(X_train, y_train)

# 予測
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# 評価
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print('=== モデルの予測性能 ===')
print(f'\n訓練データ:')
print(f'  R² スコア: {train_r2:.4f}')
print(f'  MAE: {train_mae:.2f} 千ドル')
print(f'  RMSE: {train_rmse:.2f} 千ドル')

print(f'\nテストデータ:')
print(f'  R² スコア: {test_r2:.4f}')
print(f'  MAE: {test_mae:.2f} 千ドル')
print(f'  RMSE: {test_rmse:.2f} 千ドル')

# 特徴量の重要度（係数）
print('\n=== 特徴量の重要度（係数） ===')
feature_importance = pd.DataFrame({
    '特徴量': X.columns,
    '係数': model.coef_
}).sort_values('係数', ascending=False, key=abs)
print(feature_importance)

In [ ]:
# 予測値 vs 実際の値のプロット
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 訓練データ
axes[0].scatter(y_train, y_train_pred, alpha=0.6)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('実際の価格（千ドル）')
axes[0].set_ylabel('予測価格（千ドル）')
axes[0].set_title(f'訓練データ (R² = {train_r2:.4f})')

# テストデータ
axes[1].scatter(y_test, y_test_pred, alpha=0.6)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('実際の価格（千ドル）')
axes[1].set_ylabel('予測価格（千ドル）')
axes[1].set_title(f'テストデータ (R² = {test_r2:.4f})')

plt.tight_layout()
plt.show()

## 12. まとめ

### データセットの特徴
- データ件数: 100件
- 特徴量: RM（部屋数）、LSTAT（低所得者割合）、PTRATIO（生徒-教師比率）、CRIME（犯罪率カテゴリ）
- 目的変数: PRICE（住宅価格、千ドル単位）
- 欠損値: なし

### 主な発見

1. **価格分布**:
   - 平均価格: 約22-23千ドル
   - 範囲: 約5千ドル～50千ドル
   - やや右に偏った分布

2. **犯罪率の影響**:
   - 犯罪率が低いエリアほど住宅価格が高い傾向
   - very_low > low > high の順

3. **相関分析**:
   - **RM（部屋数）**: 正の相関 - 部屋数が多いほど価格が高い
   - **LSTAT（低所得者割合）**: 負の相関 - 低所得者割合が高いほど価格が低い
   - **PTRATIO（生徒-教師比率）**: 弱い負の相関

4. **線形回帰モデル**:
   - 訓練データ R²: 約0.76-0.77
   - テストデータ R²: 約0.75-0.76
   - MAE: 約3-5千ドル
   - シンプルな線形モデルでも約76%の予測精度

### Rust実装との対応

Rustの実装では以下の追加処理を行っています：
- **特徴量エンジニアリング**: RM²（2乗項）、RM*LSTAT（交互作用項）
- **標準化**: Z-score normalization（特徴量と目的変数の両方）
- **交差検証**: 5-Fold Cross Validation

これらの追加処理により、モデルの性能と安定性が向上しています。